# West-Med CMEMS collection

Collect Copernicus Marine (CMEMS) surface currents over the western
Mediterranean (lon -6 to 20, lat 35 to 45) for **2023-06-15**, then assemble
them onto a common grid.

Sources (reanalysis products for this historical date):

- **GLORYS** global physics — total surface currents (`uo`, `vo`)
- **DUACS** global altimetry — geostrophic currents (`ugos`, `vgos`) + sea level (`sla`, `adt`)
- **MEDFS** Mediterranean physics — regional currents (`uo`, `vo`)

collekt installs the CMEMS client by default, but downloading needs Copernicus
Marine credentials: run `copernicusmarine login` once, or set
`COPERNICUSMARINE_SERVICE_USERNAME` / `COPERNICUSMARINE_SERVICE_PASSWORD`.
Files are written under `notebooks/cache/` (git-ignored); the first run
downloads, later runs reuse the cache.

In [ ]:
from pathlib import Path

import collekt
from collekt.core.config import get_config

# Western Mediterranean: west, east, south, north.
REGION = collekt.Region.from_bbox((-6.0, 20.0, 35.0, 45.0))
START = "2023-06-15"
END = "2023-06-15"
SAMPLING = "24h"


def _repo_root(marker: str = "pyproject.toml") -> Path:
    """Anchor the cache to the repo root, whatever the kernel's working dir."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    return here


OUTPUT_ROOT = _repo_root() / "notebooks" / "cache" / "westmed_cmems"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REGION.as_dict(), START, OUTPUT_ROOT

## Configure the CMEMS sources

collekt ships the adapters but no catalog, so we define the three CMEMS products
inline. `mode: auto` with a `nrt_cutoff` selects the reanalysis (`_my`) datasets
for this 2023 date and would switch to the near-real-time (`_nrt`) datasets for
recent dates.

In [ ]:
SOURCES = {
    "cmems_glorys": {
        "kind": "cmems",
        "variable_groups": ["currents"],
        "path": "cmems/glorys",
        "filename_pattern": "glorys_{dataset_id}_{date:%Y%m%d}_{bbox_hash}.nc",
        "dataset_nrt": "cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",
        "dataset_my": "cmems_mod_glo_phy_my_0.083deg_P1D-m",
        "nrt_cutoff": "2024-07-01",
        "mode": "auto",
        "variables": {"default": ["uo", "vo"]},
        "depth": [1.0, 1.1],
        "coordinates_selection_method": "inside",
    },
    "cmems_duacs": {
        "kind": "cmems",
        "variable_groups": ["currents"],
        "path": "cmems/duacs",
        "filename_pattern": "duacs_{dataset_id}_{date:%Y%m%d}_{bbox_hash}.nc",
        "dataset_nrt": "cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D",
        "dataset_my": "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D",
        "nrt_cutoff": "2024-07-01",
        "mode": "auto",
        "variables": {"default": ["ugos", "vgos"], "optional": {"sea_level": ["sla", "adt"]}},
        "use_variables": ["default", "sea_level"],
    },
    "cmems_medfs": {
        "kind": "cmems",
        "variable_groups": ["currents"],
        "path": "cmems/medfs",
        "filename_pattern": "medfs_{dataset_id}_{date:%Y%m%d}_{bbox_hash}.nc",
        "dataset_nrt": "cmems_mod_med_phy-cur_anfc_4.2km_P1D-m",
        "dataset_my": "cmems_mod_med_phy-cur_my_4.2km_P1D-m",
        "nrt_cutoff": "2024-07-01",
        "mode": "auto",
        "variables": {"default": ["uo", "vo"]},
        "depth": [1.0, 1.1],
        "coordinates_selection_method": "inside",
    },
}

config = get_config(overrides={"output": {"root": str(OUTPUT_ROOT)}, "sources": SOURCES})
list(config.sources)

## Request and download

The download is cache-aware: already-fetched files are reused. Use
`fetcher.plan()` instead of `download()` to preview the provider requests
without downloading anything.

In [ ]:
request = collekt.Request(
    region=REGION,
    start=START,
    end=END,
    sampling=SAMPLING,
    variables=("currents",),
    metadata={"region_name": "west-med"},
)

fetcher = collekt.Fetcher(request, config=config, progress=lambda source, message: print(f"[{source}] {message}"))
result = fetcher.download()

print(result.summary)
print("manifest:", result.manifest_path)
for item in result.results:
    print(f"  {item.source}: {item.status.value} -> {item.path or item.message}")

## Assemble onto a common grid

`Assembler` opens the downloaded NetCDF files and, with a non-native grid
policy, interpolates them onto a common overlapping grid. Variable names are
source-prefixed (for example `cmems_glorys__uo`) unless aliased.

In [ ]:
assembler = collekt.Assembler(result)
aliases = {
    "cmems_glorys__uo": "glorys_u",
    "cmems_glorys__vo": "glorys_v",
    "cmems_duacs__ugos": "duacs_u",
    "cmems_duacs__vgos": "duacs_v",
    "cmems_medfs__uo": "medfs_u",
    "cmems_medfs__vo": "medfs_v",
}
dataset = assembler.to_xarray(
    aliases=aliases,
    grid="lowest_resolution",
    spatial_method="linear",
    time="lowest_resolution",
    temporal_method="nearest",
    time_tolerance="1h",
)
print(sorted(str(name) for name in dataset.data_vars))
dataset

## Plot the surface-current speed

Compare the products' current-speed fields on the common grid.

In [ ]:
import matplotlib.pyplot as plt

pairs = {
    "GLORYS": ("glorys_u", "glorys_v"),
    "DUACS": ("duacs_u", "duacs_v"),
    "MEDFS": ("medfs_u", "medfs_v"),
}
available = {name: uv for name, uv in pairs.items() if uv[0] in dataset and uv[1] in dataset}

fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5), squeeze=False)
for ax, (name, (u, v)) in zip(axes[0], available.items()):
    speed = (dataset[u] ** 2 + dataset[v] ** 2) ** 0.5
    speed = speed.isel({dim: 0 for dim in speed.dims if dim not in ("latitude", "longitude")})
    speed.plot(ax=ax)
    ax.set_title(f"{name} surface-current speed (m/s)")
plt.tight_layout()
plt.show()